### Теория. Fluent Interface

**`Fluent interface` («плавный интерфейс»)** — это стиль проектирования API, при котором методы можно вызывать цепочкой, и код начинает читаться почти как фраза на естественном языке.

#### Главная идея

Чтобы работала цепочка вызовов, каждый метод возвращает объект, у которого можно вызвать следующий метод. В Python это почти всегда self.

---

#### Реализация. Способ №1

In [6]:
class Collection:
    def __init__(self, coll):
        self.coll = coll
        
    def map(self, fn):
        self.coll = list(map(fn, self.coll))
        return self
    
    def filter(self, fn):
        self.coll = list(filter(fn, self.coll))
        return self
        
    def all(self):
        return self.coll
    
cars = Collection(
    [
        {"model": "rapid", "year": 2016},
        {"model": "rio", "year": 2013},
        {"model": "mondeo", "year": 2011},
        {"model": "octavia", "year": 2014},
    ]
)

cars.filter(lambda car: car['year'] > 2013).map(lambda car: car['model'])
print(cars.all())    

['rapid', 'octavia']


#### Замечания



 **1. Отличия `self` от `self.coll`**

- `self` — это ссылка на сам экземпляр (объект) класса `Collection`. Внутри методов класса `self` позволяет обращаться к атрибутам и другим методам этого объекта. В данном коде `self` нужен, чтобы менять состояние объекта и возвращать его же (для цепочек вызовов в стиле `fluent interface`).

- `self.coll` — это атрибут экземпляра, то есть конкретное поле (переменная), в котором хранится коллекция (список) данных. В этом классе она хранит исходные элементы, над которыми выполняются операции (map и т. п.).

**2. Недостаток этого подхода**
- Сейчас `map` и `f`ilter` мутируют (изменяют) состояние объекта. Это может сбивать с толку: например, после cars.filter(...) исходный список внутри cars уже изменён.

---

### Реализация. Способ №2
Надо возвращать не `self`, a создавать новый объект того же типа с обновленной коллекцией

In [5]:
class Collection:
    def __init__(self, coll):
        self.coll = coll

    def map(self, fn):
        return Collection(list(map(fn, self.coll)))

    def filter(self, fn):
        return Collection(list(filter(fn, self.coll)))

    # Возвращает саму коллекцию, а не self.
    # Этот метод всегда последний в цепочке вызовов Collection.
    def all(self):
        return self.coll


cars = Collection(
    [
        {"model": "rapid", "year": 2016},
        {"model": "rio", "year": 2013},
        {"model": "mondeo", "year": 2011},
        {"model": "octavia", "year": 2014},
    ]
)
result = cars.filter(lambda car: car['year'] > 2013).map(lambda car: car['model'])
#filtered_сars = cars.filter(lambda car: car["year"] > 2013)
#mapped_сars = filtered_сars.map(lambda car: car["model"])
#print(mapped_сars.all())  # ['rapid', 'octavia']
print(result.all())
print(cars.all())
# [
#   {'model': 'rapid', 'year': 2016},
#   {'model': 'rio', 'year': 2013},
#   {'model': 'mondeo', 'year': 2011},
#   {'model': 'octavia', 'year': 2014}
# ]

['rapid', 'octavia']
[{'model': 'rapid', 'year': 2016}, {'model': 'rio', 'year': 2013}, {'model': 'mondeo', 'year': 2011}, {'model': 'octavia', 'year': 2014}]


#### Разбор примера
**1. Как происходит возврат из инициализатора `__init__`**

```python
def __init__(self, coll):
    self.coll = coll
```
- происходит инициализация объекта. 
**Пример для упрощения**
- допустим `c = Collection([1, 2, 3])`
- переменная `c` — это экземпляр класса, внутри которого в `self.coll` лежит список `[1, 2, 3]`.

**2. Как происходит возврат из  `map`**
```python
def map(self, fn):
    return Collection(list(map(fn, self.coll)))
```
**Здесь логика такая:**

- `self.coll` — текущий список внутри объекта.
- `map(fn, self.coll)` — вызывается встроенная функция `map` из `Python`: она применяет функцию `fn` к каждому элементу списка.
- `list(...)` превращает результат `map (итератор)` в список.
- `Collection(...)` создаёт новый экземпляр `Collection`, внутрь которого кладётся этот новый список.
- `return Collection(...)` возвращает этот новый экземпляр.

**Важный момент:**
- метод не меняет текущий объект, а создаёт и возвращает новый. Это делает цепочку безопасной: исходные данные не мутируются.

**2. Как происходит возврат из  `filter`**

```python
def filter(self, fn):
    return Collection(list(filter(fn, self.coll)))
```
Работает аналогично `map`, но использует встроенную функцию `filter`: 
- она оставляет только те элементы, для которых fn(x) возвращает `True`. 
- и тоже возвращает новый `Collection`.

**3. Как происходит возврат из  `all`**

```python
def all(self):
    return self.coll
```
Этот метод «разворачивает» коллекцию: 
- он возвращает обычный список `(self.coll)`, а не объект `Collection`. 
- Обычно его ставят в конце цепочки, чтобы получить итоговый результат в виде списка.

**4. Цепочка вызовов**

Цепочка `(fluent interface)` возможна именно потому, что `map` и `filter` возвращают объект `Collection`, у которого снова есть методы `map`, `filter`, `all`.

```python
result = cars.filter(lambda car: car['year'] > 2013).map(lambda car: car['model'])
names = Collection(['taylor', 'abigail', None])

print(result.all())
```
---

### Реализация. Способ №3

В Python `self` используется для обозначения текущего экземпляра класса. Когда вызывается `self.__class__(coll)`, создается новый экземпляр текущего класса, что идентично вызову `Collection(coll)`:

```python
class Collection:
    # ...

    def map(self, fn):
        return self.__class__(list(map(fn, self.coll)))

    # ...
```
Этот прием обеспечивает большую гибкость при наследовании классов, так как `self.__class__ `всегда ссылается на класс текущего экземпляра, а не на конкретно указанный класс.

**Что такое `self и self.__class__`**

- `self` — это ссылка на текущий экземпляр класса (объект). Внутри метода через `self` ты обращаешься к атрибутам и методам именно этого объекта.
- `self.__class__` — это класс, которому принадлежит этот экземпляр. То есть если есть `obj = Collection([1, 2, 3])`, то `obj.__class__` — это сам класс `Collection`.

Поэтому `self.__class__(coll)` — это то же самое, что Collection(coll), но только в общем виде.
Оно автоматически подставит нужный класс, даже если ты потом будешь использовать наследование.


**Как это работает внутри**

```python

return self.__class__(list(map(fn, self.coll)))
```

Происходит следующее:

- `self.__class__` возвращает класс текущего объекта.
- Вызов `self.__class__(...)` — это обычный вызов конструктора этого класса. Конструктор инициализирует новый объект, и он возвращается из метода.

*Это особенно важно для `fluent interface` (цепочки вызовов), где каждый шаг должен возвращать объект того же типа, чтобы можно было продолжать цепочку.*

---

### Задача

**Реализовать функцию `format()`** которая
- принимает на вход список городов, 
- производит внутри некоторые преобразования,
- возвращает структуру определенного формата.

**Входные данные**

```python
raw = [{'name': 'istambul', 'country': 'turkey'},
       {'name': 'Moscow ', 'country': ' Russia'},
       {'name': 'iStambul', 'country': 'tUrkey'},
       {'name': 'antalia', 'country': 'turkeY '},
       {'name': 'samarA', 'country': '  ruSsiA'}]
```

**Входная структура представляет из себя список городов:**

- каждый город это словарь с ключами `name` и `country`. 
- Значения в этих ключах не нормализованы. Они могут быть в любом регистре и содержать начальные и концевые пробелы. 
- Сами города могут дублироваться в рамках одной страны.

**Подсказки**
- Для отработки `fluent interface` в задаче используется класс `Collection`. 
- Можно использовать функции `map`, `filter` и `reduce`. Их вполне достаточно, 
- Можно поизучать функции в модуле `ds06_сollection.py`.

```python
data = [
    {'name': 'Alice', 'age': 20}, 
    {'name': 'Bob', 'age': 20}, 
    {'name': 'Alice', 'age': 20}, 
    {'name': 'Charlie', 'age': 30}
]
c = Collection(data)

c.unique().all()

# [{'age': 30, 'name': 'Charlie'},
# {'age': 20, 'name': 'Bob'},
# {'age': 20, 'name': 'Alice'}]

c.unique().group_by(lambda row: (row['age'], row['name'])).all()
# [{30: ['Charlie']}, {20: ['Bob', 'Alice']}]

c.unique().group_by(lambda row: (row['age'], row['name'])).sort_by(lambda row: list(row.keys())).all()
# [{20: ['Bob', 'Alice']}, {30: ['Charlie']}]
```
---